In [ ]:
# 경고메시지 무시
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install mglearn

# 데이터 표현

- 연속형 특성(continuous feature) : 숫자로 이루어진 순서가 있는 데이터 특성

- 범주형 특성(categorical feature) 또는 이산형 특성(dicrete feature) : 숫자가 아닌 이산적인 데이터 특성
  (컴퓨터, 옷, 책 등)
  
- 데이터의 특성보다 데이터를 어떻게 표현하는가가 머신러닝 모델의 성능에 주는 영향이 큼

-  특성 공학(feature engineering) : 특정 애플리케이션에 가장 적합한 데이터 표현을 찾는 것
  

## 레이블 인코딩(Label encoding)

- 카테고리 특성을 코드형 숫자 값으로 변환하는 것

| 상품구분 | 레이블 인코딩 |
|---|---|
|TV| 0|
|냉장고|1|
|전자렌지|2|
|컴퓨터|3|
|선풍기|4|
|믹서|5|

- 주의 : '01', '02' 코드값 역시 문자열이므로 1, 2와 같은 숫자형 값으로 변환

In [ ]:
from sklearn.preprocessing import LabelEncoder

items = ['TV','냉장고','전자렌지','컴퓨터','선풍기','믹서']

encoder = LabelEncoder()
encoder.fit(items)

print('인코딩 변환값:', encoder.transform(items) )
print('인코딩 클래스:', encoder.classes_ )
print('인코딩 원본값:', encoder.inverse_transform([4, 5, 2, 0, 1, 1]) )

- 몇몇 머신러닝 알고리즘에는 숫자 값을 적용할 경우 예측 성능이 떨어지는 경우가 발생할 수 있음
- 숫자 값의 경우 크고 작음에 대한 특성이 작용하기 때문
- 원-핫-인코딩은 레이블 인코딩의 이러한 문제점을 해결하기 위한 인코딩 방식

## 원-핫-인코딩(one-hot-encoding)

- 범주형 변수를 표현하는 데 가장 널리 쓰이는 방법
- 범주형 변수를 0 또는 1 값을 가진 하나 이상의 새로운 특성으로 바꾼 것

- 예를 들면 과목에 수학, 국어, 과학과 같이 3과목이 특성에 있다면 각각을 (1, 0, 0), (0, 1, 0) (0, 0, 1)과 같이 표현

- pandas나 scikit-learn을 이용하여 범주형 변수를 원-핫 인코딩으로 바꿀 수 있음

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

items = ['TV','냉장고','전자렌지','컴퓨터','선풍기','선풍기','믹서','믹서']

# 먼저 숫자 값으로 변환
encoder = LabelEncoder()
encoder.fit(items)
labels = encoder.transform(items)
# encoder.fit_transform(items)
print( labels.shape )

# 2차원 데이터로 변화
labels = labels.reshape(-1, 1)

# 원-핫-인코딩 적용
encoder_oh = OneHotEncoder()
encoder_oh.fit(labels)
labels_oh = encoder_oh.transform(labels)

print( labels_oh.shape )
labels_oh.toarray()

In [ ]:
# Pandas 원-핫-인코딩
# 숫자형 값으로 변환 없이도 바로 변환 가능
import pandas as pd
df = pd.DataFrame({'item':items})
pd.get_dummies(df)

### 원-핫-인코딩 실습 01

- 데이터는 1994년 인구 조사 데이터베이스에서 추출한 미국 성인 소득 데이터셋

- adult 데이터셋을 사용해 어떤 근로자의 수입이 50,000달라를 초과하는지, 그 이하일지를 예측

- 특성에는 나이, 고용형태, 교육 수준, 성별, 주당 근로시간, 직업 등이 있음

In [ ]:
import os
import pandas as pd
import mglearn

# 열 이름을 나타내는 헤더가 없으므로 header=None으로 지정 names 매개변수로 열 이름을 제공
names=['age', 'workclass', 'fnlwgt', 'education',  'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'gender',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
        'income']

# 실습을 위해 몇개의 열만 선택
select_names = ['age', 'workclass', 'education', 'gender', 'hours-per-week', 'occupation', 'income']

df = pd.read_csv( os.path.join(mglearn.datasets.DATA_PATH, 'adult.data'),
                 header=None, index_col=False, names=names )
df.head()

df = df[select_names]
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

#### 범주형 데이터 문자열 확인하기

In [ ]:
df[ df['workclass'] == ' ?' ]

In [ ]:
# 열의 내용을 확인하기 위해 유일한 값이 각각 몇 번 나타나는지 출력
df['income'].value_counts()

- 성별의 경우 Male, Female로 잘 나누어져있고, 정확하게 두 가지 값을 가지고 있어서 원-핫-인코딩으로 나타내기 좋은 형태
- 모든 열을 살펴보고 그 값들의 확인이 필요

In [ ]:
# 원 핫 인코딩 수행
pd.get_dummies(df['gender'])
df_dummies = pd.get_dummies(df)
df_dummies.head().T

In [ ]:
# 특성과 라벨을 분리
# income으로 시작하는 두 열에 인코딩된 라벨을 분리
feature = df_dummies.loc[:, 'age':'occupation_ Transport-moving']
X = feature.values
y = df_dummies['income_ >50K'].values
X.shape, y.shape

In [ ]:
# 모델 학습
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

model = LogisticRegression().fit(X_train, y_train)
model.score(X_test, y_test)

- 훈련 데이터와 테스트 데이터를 모두 담고 있는 DataFrame을 사용해 원-핫-인코딩을 하는 이유
- 훈련 세트와 테스트 세트에 범주형 값이 같은 방식으로 표현되어야 하기 때문
- 각각 가지고 있는 범주형 값에 따라 특성의 개수가 달라지는 경우가 발생

#### 숫자로 표현된 범주형 특성

- 저장 공간을 위하여 범주형 변수가 숫자로 된 경우가 많음 (남자 : 1, 여자 : 0) -> 연속형으로 다뤄야 할지 범주형으로 다뤄야 할지 어려울 수 있음

- 영화 관람 등급의 경우에는 범주형이지만 순서를 가지고 있고, 영화 별점의 경우에는 별 개수에 따라 연속형으로 나타낼 수도 있고, 범주형으로 나타낼 수도 있음.

- 풀려는 문제나 데이터 그리고 어떤 머신러닝 알고리즘을 사용할지에 따라 선택

In [ ]:
df = pd.DataFrame( {'숫자특성':[0, 1, 2, 1], '범주형특성':['양말', '여우', '양말', '우산']} )
pd.get_dummies(df)

- get_dummies 함수는 숫자 특성은 모두 연속형이라고 생각해서 가변수를 만들지 않음

#### 숫자로 표현된 범주형 특성의 원핫 인코딩

- 숫자를 문자로 변환 한 후에 원핫 인코딩
```
demo_df['숫자'] = demo_df['숫자'].astype(str)
pd.get_dummies(demo_df, columns=['숫자'])
```

In [ ]:
df['숫자특성'] = df['숫자특성'].astype(str)
pd.get_dummies(df)

## 피처 스케일링(feature scaling)

- 서로 다른 변수의 값 범위를 일정한 수준으로 맞추는 작업

- 데이터의 스케일에 매우 민감한 머신러닝 알고리즘이 존재 (SVM, 신경망 등)

- 표준화(Standardization) : 데이터의 특성 각각이 평균은 0이고 분산이 1인 가우시안 정규 분포를 가진 값으로 변환하는 것

- 정규화(Normalization) : 서로 다른 특성의 크기를 통일하기 위해 크기를 변환해주는 것

In [ ]:
# 데이터셋의 스케일을 조정하거나 전처리하는 여러 방법
mglearn.plots.plot_scaling()

### StandardScaler

- 표준화를 쉽게 지원하기 위한 클래스
- 평균이 0이고, 분산이 1인 값으로 변환

- 서포트 벡터 머신, 선형 회귀, 로지스틱 회귀는 데이터가 가우시안 분포를 가지고 있다고 가정하고 구현되었기에 표준화 적용이 필요

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target,
                                                    random_state=0)
X_train.shape, X_test.shape

In [ ]:
from sklearn.svm import SVC

model = SVC(C=100).fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
# 평균 0, 분산 1을 갖도록 스케일 조정
from sklearn.preprocessing import StandardScaler

# 주의 : 변환을 위한 기준 정보 설정
# 학습 데이터와 테스트 데이터의 스케일링 기준 정보를 동일하게 유지
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 조정된 데이터로 SVM 학습
# 스케일 조정된 테스트 세트의 정확도
model_s = SVC(C=100).fit(X_train_scaled, y_train)
model_s.score(X_test_scaled, y_test)

In [ ]:
import pandas as pd

df = pd.DataFrame(X_train_scaled)
print("특성들의 평균 값: \n", df.mean().max(), df.mean().min() )
print("특성들의 분산 값: \n", df.var().max(), df.var().min() )

- 특성의 평균이 0에 아주 가까운 값으로, 분산은 1에 아주 가까운 값으로 변환

### MinMaxScaler

- 정규화를 쉽게 지원하기 위한 클래스
- 데이터값을 0과 1사이의 범위 값으로 변환


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 0~1 사이로 스케일 조정
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_mms = scaler.transform(X_train)
X_test_mms = scaler.transform(X_test)

# 조정된 데이터로 SVM 학습
# 스케일 조정된 테스트 세트의 정확도
model_mms = SVC(C=100).fit(X_train_mms, y_train)
model_mms.score(X_test_mms, y_test)

In [ ]:
# 특성들의 최소값
# 특성들의 최대값
df = pd.DataFrame(X_train_mms)
print("특성들의 평균 값: \n", df.mean().max(), df.mean().min() )
print("특성들의 분산 값: \n", df.var().max(), df.var().min() )
# print("특성들의 최대/최소 값: \n", df.max(), df.min() )

# 특성 공학(feature engineering)

- 특정 애플리케이션에 가장 적합한 데이터 표현을 찾는 것

- 데이터를 가장 잘 표현하는 방법은 데이터가 가진 의미뿐 아니라 어떤 모델을 사용하는지에 따라 다름

 - 구간분할(bining) : 한 특성을 여러 특성으로 나누는 것 (이산화)
 ```
 from sklearn.preprocessing import KBinsDiscretizer
 ```

 - 용량이 매우 크고 고차원 데이터셋이라 선형 모델을 사용해야 한다면 구간 분할이 모델 성능을 높이는데 도움이 됨

- 원본 데이터에 상호작용(interaction)과 다항식(polyonmial) 특성을 추가

## 특성 자동 선택

- 특성이 추가되면 모델이 복잡해지고 과대적합 가능성이 올라감

- 유용한 특성만 선택하고 나머지는 무시해서 특성의 수를 줄이는 것이 좋음.

- 특성 선택 방법 : 일변량 통계, 모델 기반 선택, 반복적 선택


### 일변량 통계(univariate statistics)

- 특성과 타켓 사이에 중요한 통계적 관계가 있는지 계산하고 깊게 관련된 것만 선택

In [ ]:
# 의미 없는 노이즈 특성을 추가하고, 식별해서 제거하는지 확인
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

cancer = load_breast_cancer()

# 고정된 난수를 발생
rn = np.random.RandomState(42)
noise = rn.normal( size=(len(cancer.data), 50) )
print( noise.shape )

# 데이터에 노이즈 특성을 추가
# 처음 30개는 원본 특성이고 다음 50개는 노이즈
X_noise = np.hstack([cancer.data, noise])
print( X_noise.shape )

X_train, X_test, y_train, y_test = train_test_split(X_noise, cancer.target,
                                                    random_state=0,
                                                    test_size=0.5)
print( X_train.shape )

# SelectPercentile을 사용하여 특성의 50%를 선택
select = SelectPercentile(percentile=50)
select.fit(X_train, y_train)

X_train_selected = select.transform(X_train)
X_train_selected.shape

In [ ]:
# 선택된 특성을 boolen 값으로 표시
# True는 검은색, False는 흰색으로 마스킹
mask = select.get_support()
plt.matshow( mask.reshape(1, -1), cmap='gray_r' )
plt.yticks([0])
plt.show()

- 선택된 특성은 대부분 원본 특성이고 노이즈 특성이 거의 모두 제거

전체 특성을 이용했을 때와 선택된 특성만 사용했을 때의 성능 비교

In [ ]:
from sklearn.linear_model import LogisticRegression

# 테스트 데이터 변환
X_test_selected = select.transform(X_test)

# 전체 특성을 사용한 점수
model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
# 선택된 일부 특성을 사용한 점수
model = LogisticRegression(solver='liblinear')
model.fit(X_train_selected, y_train)
model.score(X_test_selected, y_test)

- 일부 원본 특성이 없더라도 노이즈 특성을 제거한 쪽의 성능이 더 높음
- 너무 많은 특성 때문에 모델을 만들기가 현실적으로 어려울 때 사용
- 많은 특성들이 확실히 도움이 안 된다고 생각될 때 사용

### 모델 기반 선택(model-based selection)

- 지도 학습 머신러닝 모델을 사용하여 특성의 중요도를 평가해서 가장 중요한 것만 선택

- 지도학습 모델은 최종적으로 사용할 학습 모델과 같을 필요는 없음

- 특성 선택을 위한 모델은 각 특성의 중요도를 측정하여 순서 매길 수 있어야 함

- 결정트리 기반의 모델들은 각 특성의 중요도를 feature_importances_에 저장

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# (지도 학습 모델로 계산된) 중요도가 지정한 임계치보다 큰 모든 특성 선택
# 일변량 분석으로 선택한 특성과 결과를 비교하기 위해
# 절반 가량의 특성이 선택될수 있도록 중간값을 임계치로 사용
select = SelectFromModel(RandomForestClassifier(random_state=0),
                         threshold='median')
select.fit(X_train, y_train)

# 선택된 특성을 boolen 값으로 표시
# True는 검은색, False는 흰색으로 마스킹
mask = select.get_support()
plt.matshow( mask.reshape(1, -1), cmap='gray_r' )
plt.yticks([0])
plt.show()
X_train_m = select.transform(X_train)
X_train_m.shape

- 네 개를 제외한 모든 원본 특성이 선택 (특성을 40개 선택 -> 일부 노이즈도 선택)



In [ ]:
# 예측 성능 확인
X_train_m = select.transform(X_train)
X_test_m = select.transform(X_test)

model = LogisticRegression(solver='liblinear')
model.fit(X_train_m, y_train)
model.score(X_test_m, y_test)

- 특성 선택이 더 잘 되었으므로 성능도 조금 향상

### 반복적 특성 선택(iterative selection)

- 특성의 수가 각기 다른 모델에서 만들어짐

- 방법

  1. 특성을 하나도 선택하지 않은 상태로 시작해 어떤 종료 조건을 도달할 때까지 하나씩 추가하는 방법
  2. 모든 특성을 가지고 시작해 어떤 종료 조건이 될 때까지 하나씩 제거하는 방법

In [ ]:
from sklearn.feature_selection import RFE

select = RFE(RandomForestClassifier(random_state=0), n_features_to_select=20)
select.fit(X_train, y_train)

# 선택된 특성을 boolen 값으로 표시
# True는 검은색, False는 흰색으로 마스킹
mask = select.get_support()
plt.matshow( mask.reshape(1, -1), cmap='gray_r' )
plt.yticks([0])
plt.show()

- 이전 보다 특성 선택이 나아졌지만, 여전히 특성 두 개를 놓침
- 랜덤 포레스트 모델은 특성이 누락될 때마다 다시 학습하므로 40번이나 실행하여 모델 기반 선택보다 오래 걸림

In [ ]:
# 예측 성능 확인
X_train_rfe = select.transform(X_train)
X_test_rfe = select.transform(X_test)

model = LogisticRegression(solver='liblinear')
model.fit(X_train_rfe, y_train)
model.score(X_test_rfe, y_test)

- 특성 선택이 제대로 되면 선형 모델의 성능은 랜덤 포레스트와 견줄만 함

## 전문가적 지식 활용

- 특성 공학은 특정 어플리케이션을 위해 전문가적 지식을 사용할 수 있는 중요한 영역
- 분야 전문가는 종종 초기 데이터에서 더 유용한 특성을 선택할 수 있도록 도움을 줄 수 있음

- 해당 분야에 내재된 사전 지식이 특성으로 추가될 수 있다면 머신러닝 알고리즘에 도움이 됨

## 특성 공학 실습 01

시티바이크 데이터

- http://www.citibikenyc.com/system-data

- 문제 : 특정 날짜와 시간에 앤디 집 앞에 있는 자전거를 사람들이 얼마나 대여할 것인지 예측하는 것
- 앤디가 빌릴 자전거가 남아 있을까?

차트 한글

```
# Windows
from matplotlib import font_manager, rc

font_name = font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family=font_name)
```
```
from matplotlib import rc
rc('font', family='Malgun Gothic')
```

In [ ]:
# 2015년 8월 데이터 세 시간 간격 일일 주요 경향
import mglearn

citibike = mglearn.datasets.load_citibike()

type(citibike)
citibike.head()

In [ ]:
# 8월 한 달 동안의 대여 횟수
import pandas as pd

xticks = pd.date_range(start=citibike.index.min(), end=citibike.index.max(), freq='D' )
week = ['일', '월', '화', '수', '목', '금', '토']
xticks_name = [ week[int(w)]+d for w, d in zip(xticks.strftime('%w'), xticks.strftime(' %m-%d')) ]

plt.figure( figsize=(15, 4) )
plt.plot(citibike)
plt.xticks(xticks, labels=xticks_name, rotation=90)
plt.show()

- 낮과 밤, 주중과 주말의 패턴 차이


#### 훈련 세트와 테스트 세트로 나누기

- 시계열 데이터를 이용한 예측 작업은 과거 데이터에서 학습하여 미래를 예측하는 방식
- 어떤 날짜를 기준으로 훈련 세트와 테스트 세트를 나눔

In [ ]:
# 3시간 누적 카운트 이므로 하루 24/3=8 개씩 23일 치는 184개 8일 치는 64개
# 처음 184개 데이터 포인트를 훈련 세트로 사용하고 나머지는 테스트 세트로 사용
n_train = 184

# 주어진 특성을 사용하여 평가하고 그래프를 만듬
def eval_on_features(features, target, regressor):
    # 훈련 세트와 테스트 세트로 나눔
    X_train, X_test = features[:n_train], features[n_train:]
    y_train, y_test = target[:n_train], target[n_train:]

    regressor.fit(X_train, y_train)
    print("테스트 세트 R^2: {:.2f}".format(regressor.score(X_test, y_test)))

    y_pred = regressor.predict(X_test)
    y_pred_train = regressor.predict(X_train)
    plt.figure(figsize=(10, 3))

    plt.xticks(range(0, len(X), 8), xticks_name, rotation=90, ha="left")

    plt.plot(range(n_train), y_train, label="train")
    plt.plot(range(n_train, len(y_test) + n_train), y_test, '-', label="test")
    plt.plot(range(n_train), y_pred_train, '--', label="train predict")

    plt.plot(range(n_train, len(y_test) + n_train), y_pred, '--', label="test predict")
    plt.legend(loc=(1.01, 0))
    plt.xlabel("date")
    plt.ylabel("rental number")

#### 첫 번째 시도는 날짜와 시간을 하나의 숫자로 표현한 특성을 사용

In [ ]:
# 타깃값 추출 (대여 횟수)
y = citibike.values

# POSIX 시간 : 1970/1/1 00:00:00 부터 초로 나타냄
# POSIX 시간을 10**9로 나누어 변환
X = citibike.index.astype('int64').values.reshape(-1, 1) // 10**9

RandomForestRegressro 이용

- 데이터 전처리가 거의 필요하지 않아 처음 시도해보기 좋은 모델

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=0)
eval_on_features(X, y, model)

- 랜덤 포레스트는 훈련 세트에 있는 특성의 범위 밖으로 외샵(extrapolation)할 수 있는 능력이 없음
- 테스트 세트와 가장 가까이 있는 마지막 훈련 세트 데이터의 타깃값을 예측으로 사용

#### 두 번째 시도는 시간과 요일 특성을 추가

- 전문가적 지식 : 훈련 데이터의 대여 데이터를 시간과 요일 이라는 두 요소가 중요한 것으로 보임

In [ ]:
# 시간 정보 추가
X_hour = citibike.index.hour.values.reshape(-1, 1)
eval_on_features(X_hour, y, model)

In [ ]:
# 요일 정보도 추가
X_week = citibike.index.dayofweek.values.reshape(-1, 1)
eval_on_features(X_week, y, model)

In [ ]:
X_hour_week = np.hstack( [X_week, X_hour] )
eval_on_features(X_hour_week, y, model)

In [ ]:
# LinearRegression 을 적용
# 굳이 랜덤 포레스트같이 복잡한 모델이 필요하지 않음
from sklearn.linear_model import LinearRegression
eval_on_features(X_hour_week, y, LinearRegression())

- 성능 나쁘고 주기 패턴 이상 이유
 - 요일과 시간이 정수로 인코딩되어 있어서 연속형 변수로 해석되기 때문
 - 하루에서 시간이 흐를 수록 대여수가 늘어나게 학습되어 있음
 - 하지만 실제 패턴 이보다 복잡

- 이 패턴을 잡아내기 위해 OneHotEncoder를 사용하여 정수형을 범주형 변수로 해석

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
X_hour_week_oh = encoder.fit_transform(X_hour_week).toarray()
X_hour_week_oh.shape

Ridge 이용

In [ ]:
from sklearn.linear_model import Ridge
eval_on_features(X_hour_week_oh, y, Ridge())

- 연속형 특성일 때보다 훨씬 좋아짐

 - 이 선형 모델은 요일에 대해 하나의 계수를 학습
 - 시간에 대해서도 하나의 계수 학습
 - 시간 패턴이 모든 날에 걸쳐 공유된다는 뜻

- 상호작용 특성을 사용하면 시간과 요일 조합별 계수를 학습 가능

In [ ]:
# 상호작용 특성 추가
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_hour_week_oh_poly = poly.fit_transform(X_hour_week_oh)
X_hour_week_oh_poly.shape

In [ ]:
lr = Ridge()
eval_on_features(X_hour_week_oh_poly, y, lr)

- 이 모델 가장 큰 장점 : 무엇이 학습되었는지 명확

- 각 날짜와 시간에 대해 하나의 계수를 학습

- 랜덤 포레스트와는 달리 이 모델이 학습한 계수를 그래프로 나타낼수 있음

In [ ]:
# 시간과 요일 특성의 이름
hour = ["%02d:00" % i for i in range(0, 24, 3)]
day = ["월", "화", "수", "목", "금", "토", "일"]
features =  day + hour

In [ ]:
# 추출한 모든 상호작용 특성에 이름을 부여
# 계수가 0이 아닌 특성만 선택
features_poly = poly.get_feature_names(features)
features_nonzero = np.array(features_poly)[lr.coef_ != 0]
coef_nonzero = lr.coef_[lr.coef_ != 0]

In [ ]:
# 시간과 요일의 곱을 사용한 선형 모델의 계수
plt.figure(figsize=(15, 2))
plt.plot(coef_nonzero, 'o')
plt.xticks(np.arange(len(coef_nonzero)), features_nonzero, rotation=90)
plt.xlabel("특성 이름")
plt.ylabel("계수 크기")
plt.title("시간과 요일의 곱을 사용한 선형 모델의 계수(가중치)")
plt.show()

## 특성 공학 실습 02

캐글 주택 가격

- https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

- 문제 : 주택 판매 가격 예측


### 데이터 사전 처리(Preprocessing)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('house_price.csv')
df.head()

In [ ]:
# 결측치
# df.info()
# df.describe()
isnull_df = df.isnull().sum()
isnull_df[ isnull_df > 0 ].sort_values(ascending=False)

In [ ]:
# y 분포
# df['SalePrice'].hist()
sns.distplot(df['SalePrice'])

In [ ]:
sns.distplot( np.log1p(df['SalePrice']) )

In [ ]:
df['SalePrice'] = np.log1p(df['SalePrice'])

In [ ]:
# 특성들의 분포(보류)
df['GrLivArea'].hist()

In [ ]:
# Id와 너무 많은 결측치를 가지고 있는 특성 제거
df.drop( ['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu'],
        axis=1, inplace=True)
df.shape

In [ ]:
# 연속형 특성 결측 채움
df.fillna( df.mean(), inplace=True)

In [ ]:
isnull_df = df.isnull().sum()
isnull_df[ isnull_df > 0 ].sort_values(ascending=False)

In [ ]:
df_oh = pd.get_dummies(df)
df.shape, df_oh.shape

### 선형 회귀 모델의 학습/예측/평가

In [ ]:
def get_rmse(model):
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test , pred)
    rmse = np.sqrt(mse)
    print('{0} 로그 변환된 RMSE: {1}'.format(model.__class__.__name__,np.round(rmse, 3)))
    return rmse

def get_rmses(models):
    rmses = [ ]
    for model in models:
        rmse = get_rmse(model)
        rmses.append(rmse)
    return rmses

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

X = df_oh.drop('SalePrice', axis=1)
y = df_oh['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

tree_reg = RandomForestRegressor(random_state=0).fit(X_train, y_train)
lr_reg = LinearRegression().fit(X_train, y_train)
ridge_reg = Ridge().fit(X_train, y_train)
lasso_reg = Lasso().fit(X_train, y_train)

get_rmses( [tree_reg, lr_reg, ridge_reg, lasso_reg] )

In [ ]:
from sklearn.model_selection import cross_val_score

# 분할하지 않고 전체 데이터로 cross_val_score( ) 수행
# 모델별 CV RMSE값과 평균 RMSE 출력
def get_avg_rmse_cv(models):
    pass


In [ ]:
# lr_reg, ridge_reg, lasso_reg 모델의 CV RMSE값 출력
models = [tree_reg, lr_reg, ridge_reg, lasso_reg]
for model in models:
  mse_list = -1 * cross_val_score(model, X, y,
                                  scoring='neg_mean_squared_error', cv=5)
  rmse_list = np.sqrt(mse_list)
  rmse_avg = rmse_list.mean()
  print(f'{model.__class__.__name__} CV RMSE list: {np.round(rmse_list, 3)}')
  print(f'{model.__class__.__name__} CV RMSE AVG: {rmse_avg:.3f}')
  print()


* 하이퍼 파라미터 튜닝

In [ ]:
from sklearn.model_selection import GridSearchCV

def print_best_params(model, params):
  grid_search = GridSearchCV(model, param_grid=params,
                             scoring='neg_mean_squared_error', cv=5)
  grid_search.fit(X, y)
  rmse = np.sqrt( -1 * grid_search.best_score_ )
  print(f'{model.__class__.__name__} CV RMSE AVG: {rmse:.3f}',
        f' Params: {grid_search.best_params_}')

  return grid_search.best_estimator_


In [ ]:
# 최적화 alpha값으로 학습, 예측 및 평가 수행.
param_ridge = { 'alpha':[0.05, 0.1, 1, 5, 8, 10, 12, 15, 20] }
param_lasso = { 'alpha':[0.001, 0.005, 0.008, 0.05, 0.03, 0.1, 0.5, 1, 5, 10] }
print_best_params(Ridge(), param_ridge)
print_best_params(Lasso(), param_lasso)

In [ ]:
lr_reg = LinearRegression().fit(X_train, y_train)
ridge_reg = Ridge(alpha=12).fit(X_train, y_train)
lasso_reg = Lasso(alpha=0.001).fit(X_train, y_train)

get_rmses( [lr_reg, ridge_reg, lasso_reg] )

* 데이터 왜곡 로그 변환 후 성능 평가

In [ ]:
from scipy.stats import skew

f_index = df.dtypes[ df.dtypes != 'object' ].index
skew_features = df[f_index].apply(lambda x: skew(x))

skew_features_top = skew_features[ skew_features > 1 ]
skew_features_top.sort_values(ascending=False)

In [ ]:
df['GrLivArea'].hist()

In [ ]:
df[skew_features_top.index] = np.log1p( df[skew_features_top.index] )

In [ ]:
df['GrLivArea'].hist()

In [ ]:
df_oh = pd.get_dummies(df)
df.shape, df_oh.shape

X = df_oh.drop('SalePrice', axis=1)
y = df_oh['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

param_ridge = { 'alpha':[0.05, 0.1, 1, 5, 8, 10, 12, 15, 20] }
param_lasso = { 'alpha':[0.001, 0.005, 0.008, 0.05, 0.03, 0.1, 0.5, 1, 5, 10] }
ridge_reg = print_best_params(Ridge(), param_ridge)
lasso_reg = print_best_params(Lasso(), param_lasso)

In [ ]:
get_rmses( [ridge_reg, lasso_reg] )

* 이상치 제거

In [ ]:
df_org = pd.read_csv('house_price.csv')
plt.scatter(df_org['GrLivArea'], df_org['SalePrice'])

In [ ]:
con1 = df_oh['GrLivArea'] > np.log1p(4000)
con2 = df_oh['SalePrice'] < np.log1p(500000)
outlier_index =  df_oh[ con1 & con2 ].index
df_oh.drop(outlier_index, axis=0, inplace=True)
df_oh.shape

In [ ]:
X = df_oh.drop('SalePrice', axis=1)
y = df_oh['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

param_ridge = { 'alpha':[0.05, 0.1, 1, 5, 8, 10, 12, 15, 20] }
param_lasso = { 'alpha':[0.001, 0.005, 0.008, 0.05, 0.03, 0.1, 0.5, 1, 5, 10] }
ridge_reg = print_best_params(Ridge(), param_ridge)
lasso_reg = print_best_params(Lasso(), param_lasso)

In [ ]:
get_rmses( [ridge_reg, lasso_reg] )

### 회귀 트리 학습/예측/평가

In [ ]:
from xgboost import XGBRegressor


In [ ]:
from lightgbm import LGBMRegressor


### 회귀 모델들의 예측 결과 혼합을 통한 최종 예측

In [ ]:
pred_ridge = ridge_reg.predict(X_test)
pred_lasso = lasso_reg.predict(X_test)

pred = (0.6 * pred_ridge) + (0.4 * pred_lasso)
rmse = np.sqrt( mean_squared_error(y_test, pred) )
rmse